# GCS short approach and same-counter pouring

This notebook keeps the pitcher and bowl on the kitchen island. The PR2 uses GCS for one short base approach, picks up the pitcher, and pours into the nearby bowl without navigating again.

Pouring is represented by aligning and tilting the pitcher over the bowl; liquid particles are not simulated. Run the notebook from top to bottom with the `new-cram-env` kernel.


In [1]:
# %% Collapsed demo infrastructure
import math
import threading
import time
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import rclpy
from rclpy.node import Node
from scipy.spatial.transform import Rotation

import coraplex
from coraplex.datastructures.dataclasses import Context
from coraplex.datastructures.enums import Arms, ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.execution_environment import simulated_robot
from coraplex.locations.factories import reachability_location
from coraplex.plans.factories import code, sequential
from coraplex.robot_plans.actions.composite.tool_based import PouringAction
from coraplex.robot_plans.actions.core.navigation import NavigateAction
from coraplex.robot_plans.actions.core.pick_up import PickUpAction
from coraplex.robot_plans.actions.core.placing import PlaceAction
from coraplex.robot_plans.actions.core.robot_body import MoveTorsoAction, ParkArmsAction
from coraplex.testing import setup_world
from coraplex.view_manager import ViewManager
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.datastructures.definitions import TorsoState
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Bowl, Milk, PouringCup, Table
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Point3, Pose
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.geometry import VolumetricBoundingBox
from semantic_digital_twin.world_description.graph_of_convex_sets.base import GraphOfConvexSets
from semantic_digital_twin.world_description.graph_of_convex_sets.boxes import VolumetricGraphOfBoundingBoxes
from semantic_digital_twin.world_description.shape_collection import BoundingBoxCollection
from semantic_digital_twin.world_description.world_entity import Body


@dataclass(frozen=True)
class FixedKitchen:
    """Scene state shared by fixed-kitchen demonstrations."""

    world: World
    """Apartment world used by the demonstration."""

    robot: PR2
    """Robot annotation bound to the apartment world."""

    initial_base_pose: Pose
    """Robot base pose before preparation or navigation."""


@dataclass(frozen=True)
class NavigationRoute:
    """Reachable base pose and the GCS path leading to it."""

    base_pose: Pose
    """Reachable robot base pose at the route destination."""

    navigation_point: Point3
    """Destination projected onto the navigation layer."""

    path: list[Point3]
    """Filtered GCS path from the preceding route endpoint."""


def setup_fixed_kitchen(
    initial_base_position: tuple[float, float] | None = None,
) -> FixedKitchen:
    """Create the fixed apartment and bind its PR2 annotation."""
    world = setup_world()
    robot = PR2.from_world(world)
    if initial_base_position is not None:
        position_x, position_y = initial_base_position
        with world.modify_world():
            robot.root.parent_connection.origin = (
                HomogeneousTransformationMatrix.from_xyz_rpy(
                    x=position_x,
                    y=position_y,
                    reference_frame=world.root,
                )
            )
    return FixedKitchen(
        world=world,
        robot=robot,
        initial_base_pose=robot.root.global_pose,
    )


def start_ros_node(node_name: str) -> tuple[Node, threading.Thread]:
    """Start a ROS node and retain its background spinning thread."""
    if not rclpy.ok():
        rclpy.init()
    ros_node = rclpy.create_node(node_name)
    ros_thread = threading.Thread(target=rclpy.spin, args=(ros_node,), daemon=True)
    ros_thread.start()
    return ros_node, ros_thread


def start_visualization_publisher(
    world: World,
    ros_node: Node,
) -> tuple[TFPublisher, VizMarkerPublisher]:
    """Start synchronized transform and marker publication for a world."""
    tf_publisher = TFPublisher(_world=world, node=ros_node)
    visualization_publisher = VizMarkerPublisher(_world=world, node=ros_node)
    return tf_publisher, visualization_publisher


def create_navigation_graph(
    kitchen: FixedKitchen,
    bounds: tuple[float, float, float, float],
    region_name: str,
    navigation_height: float = 0.2,
    obstacle_bloat: float = 0.22,
) -> GraphOfConvexSets:
    """Build and expose the fixed kitchen's GCS navigation free space."""
    minimum_x, maximum_x, minimum_y, maximum_y = bounds
    search_space = BoundingBoxCollection(
        [
            VolumetricBoundingBox(
                min_x=minimum_x,
                max_x=maximum_x,
                min_y=minimum_y,
                max_y=maximum_y,
                min_z=navigation_height - 0.05,
                max_z=navigation_height + 0.05,
                origin=HomogeneousTransformationMatrix(reference_frame=kitchen.world.root),
            )
        ],
        reference_frame=kitchen.world.root,
    )
    navigation_graph = VolumetricGraphOfBoundingBoxes.free_space_from_world(
        world=kitchen.world,
        search_space=search_space,
        bloat_obstacles=obstacle_bloat,
    )
    # create_as_region moved to HasSupportingSurface.spawn_bounding_boxes_as_region
    # upstream; dropped here since nothing downstream consumes the spawned Region --
    # navigation_graph itself (path_from_to / plot_free_space) is what's actually used.
    return navigation_graph


def navigation_point(
    base_pose: Pose,
    reference_frame: Body,
    navigation_height: float = 0.2,
) -> Point3:
    """Project a robot base pose onto the GCS navigation layer."""
    return Point3(
        x=base_pose.x,
        y=base_pose.y,
        z=navigation_height,
        reference_frame=reference_frame,
    )


def filter_navigation_path(
    raw_path: list[Point3],
    minimum_waypoint_distance: float = 0.08,
) -> list[Point3]:
    """Remove consecutive decomposition waypoints below the requested distance."""
    filtered_path = [raw_path[0]]
    for candidate in raw_path[1:-1]:
        previous = filtered_path[-1]
        distance = math.hypot(
            float(candidate.x - previous.x),
            float(candidate.y - previous.y),
        )
        if distance >= minimum_waypoint_distance:
            filtered_path.append(candidate)
    filtered_path.append(raw_path[-1])
    return filtered_path


def create_navigation_route(
    navigation_graph: GraphOfConvexSets,
    start_point: Point3,
    destination_point: Point3,
    base_pose: Pose,
    minimum_waypoint_count: int = 1,
) -> NavigationRoute:
    """Create a filtered route to a fixed, known navigation target."""
    raw_path = navigation_graph.path_from_to(start_point, destination_point)
    if raw_path is None or len(raw_path) < minimum_waypoint_count:
        raise RuntimeError("The fixed navigation target is not connected")
    return NavigationRoute(
        base_pose=base_pose,
        navigation_point=destination_point,
        path=filter_navigation_path(raw_path),
    )


def find_optional_reachable_route(
    navigation_graph: GraphOfConvexSets,
    start_point: Point3,
    target: Body | Pose,
    kitchen: FixedKitchen,
    context: Context,
    arm: Arms,
    grasp_description: GraspDescription,
    navigation_height: float = 0.2,
) -> NavigationRoute | None:
    """Return the first connected reachable target pose, when one exists."""
    for candidate_base_pose in reachability_location(
        target,
        context,
        arm,
        grasp_description,
    ):
        candidate_point = navigation_point(
            candidate_base_pose,
            kitchen.world.root,
            navigation_height,
        )
        candidate_path = navigation_graph.path_from_to(start_point, candidate_point)
        if candidate_path is not None:
            return NavigationRoute(
                base_pose=candidate_base_pose,
                navigation_point=candidate_point,
                path=filter_navigation_path(candidate_path),
            )
    return None


def find_reachable_route(
    navigation_graph: GraphOfConvexSets,
    start_point: Point3,
    target: Body | Pose,
    kitchen: FixedKitchen,
    context: Context,
    arm: Arms,
    grasp_description: GraspDescription,
    navigation_height: float = 0.2,
) -> NavigationRoute:
    """Return a connected reachable target pose or report that none exists."""
    route = find_optional_reachable_route(
        navigation_graph,
        start_point,
        target,
        kitchen,
        context,
        arm,
        grasp_description,
        navigation_height,
    )
    if route is None:
        raise RuntimeError(
            "No reachable target pose is connected to the navigation graph"
        )
    return route


def interpolate_navigation_path(
    path: list[Point3],
    final_yaw: float,
    reference_frame: Body,
    step_distance: float = 0.08,
) -> list[Pose]:
    """Convert a GCS path into closely spaced robot base poses."""
    poses = []
    for segment_start, segment_end in zip(path, path[1:]):
        start_x = float(segment_start.x)
        start_y = float(segment_start.y)
        distance_x = float(segment_end.x) - start_x
        distance_y = float(segment_end.y) - start_y
        segment_distance = math.hypot(distance_x, distance_y)
        step_count = max(1, math.ceil(segment_distance / step_distance))
        segment_yaw = math.atan2(distance_y, distance_x)
        for step_index in range(1, step_count + 1):
            progress = step_index / step_count
            poses.append(
                Pose.from_xyz_rpy(
                    x=start_x + distance_x * progress,
                    y=start_y + distance_y * progress,
                    z=0.0,
                    yaw=segment_yaw,
                    reference_frame=reference_frame,
                )
            )
    final_point = path[-1]
    poses[-1] = Pose.from_xyz_rpy(
        x=final_point.x,
        y=final_point.y,
        z=0.0,
        yaw=final_yaw,
        reference_frame=reference_frame,
    )
    return poses


def create_navigation_playback(
    route: NavigationRoute,
    reference_frame: Body,
    seconds_per_step: float = 0.08,
) -> tuple[list[Pose], list[object]]:
    """Create visible navigation poses and their sequential plan nodes."""
    final_yaw = float(route.base_pose.to_quaternion().to_rpy()[2])
    poses = interpolate_navigation_path(route.path, final_yaw, reference_frame)
    steps = []
    for pose in poses:
        steps.extend(
            [
                NavigateAction(pose),
                code(partial(time.sleep, seconds_per_step)),
            ]
        )
    return poses, steps


def plot_navigation_routes(
    navigation_graph: GraphOfConvexSets,
    routes: list[tuple[str, NavigationRoute, str]],
    target: Point3 | None = None,
    navigation_height: float = 0.2,
) -> go.Figure:
    """Plot the GCS free space and named routes for notebook inspection."""
    figure = go.Figure(navigation_graph.plot_free_space())
    for route_index, (name, route, color) in enumerate(routes):
        figure.add_trace(
            go.Scatter3d(
                x=[point.x for point in route.path],
                y=[point.y for point in route.path],
                z=[navigation_height + route_index * 0.01 for point in route.path],
                mode="lines+markers",
                name=name,
                line={"color": color, "width": 8},
            )
        )
    if target is not None:
        figure.add_trace(
            go.Scatter3d(
                x=[target.x],
                y=[target.y],
                z=[target.z],
                mode="markers",
                name="Task target",
                marker={"color": "green", "size": 8},
            )
        )
    figure.update_layout(height=600, scene_aspectmode="data")
    return figure


def front_grasp(kitchen: FixedKitchen, arm: Arms) -> GraspDescription:
    """Create the common front grasp used by fixed-kitchen examples."""
    return GraspDescription(
        approach_direction=ApproachDirection.FRONT,
        vertical_alignment=VerticalAlignment.NoAlignment,
        end_effector=ViewManager.get_end_effector_view(arm, kitchen.robot),
    )


def placement_pose_from_surface_sample(
    sampled_point: Point3,
    table: Table,
    object_body: Body,
    world: World,
) -> Pose:
    """Create an object pose from a sampled free table-surface point."""
    sampled_world_point = world.transform(sampled_point, world.root)
    table_top_height = float(table.root.global_pose.z + table.min_max_points[1].z)
    object_half_height = float(object_body.combined_mesh.extents[2] / 2)
    orientation = world.transform(
        Pose(reference_frame=table.supporting_surface),
        world.root,
    ).to_quaternion()
    return Pose.from_xyz_quaternion(
        sampled_world_point.x,
        sampled_world_point.y,
        table_top_height + object_half_height,
        *orientation.to_list(),
        reference_frame=world.root,
    )


## Spawn a nearby pitcher and bowl

Both vessels are placed on the grocery island using the island surface height. The robot starts close to the known pouring pose so the GCS route is only a short approach.


In [2]:
kitchen = setup_fixed_kitchen(initial_base_position=(1.45, 2.32))
world, robot = kitchen.world, kitchen.robot

object_directory = Path(coraplex.__file__).resolve().parents[2] / "resources" / "objects"
pitcher_world = STLParser(str(object_directory / "Static_MilkPitcher.stl")).parse()
bowl_world = STLParser(str(object_directory / "bowl.stl")).parse()
milk = world.get_body_by_name("milk.stl")
island_height = float(milk.global_pose.z + milk.combined_mesh.bounds[0][2])
pitcher_height = island_height - float(pitcher_world.root.combined_mesh.bounds[0][2])
bowl_height = island_height - float(bowl_world.root.combined_mesh.bounds[0][2])

with world.modify_world():
    world.merge_world_at_pose(
        pitcher_world,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            x=2.37,
            y=2.2,
            z=pitcher_height,
            reference_frame=world.root,
        ),
    )
    world.merge_world_at_pose(
        bowl_world,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            x=2.4,
            y=2.45,
            z=bowl_height,
            reference_frame=world.root,
        ),
    )

pitcher = world.get_body_by_name("Static_MilkPitcher.stl")
bowl = world.get_body_by_name("bowl.stl")
pouring_source = PouringCup(root=pitcher)
with world.modify_world():
    world.add_semantic_annotations([pouring_source, Bowl(root=bowl)])

{
    "initial_base": [round(float(value), 3) for value in kitchen.initial_base_pose.to_position().to_list()[:3]],
    "pitcher_position": [round(float(value), 3) for value in pitcher.global_pose.to_position().to_list()[:3]],
    "bowl_position": [round(float(value), 3) for value in bowl.global_pose.to_position().to_list()[:3]],
}


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


{'initial_base': [1.45, 2.32, 0.0],
 'pitcher_position': [2.37, 2.2, 1.022],
 'bowl_position': [2.4, 2.45, 0.995]}

In [3]:
ros_node, ros_thread = start_ros_node("coraplex_gcs_kitchen_counter_pouring")
context = Context(world=world, robot=robot, ros_node=ros_node)


## Publish the world

TF starts before MarkerArray publication so RViz can transform the vessels immediately. Use the printed fixed frame and marker topic.


In [4]:
tf_publisher, visualization_publisher = start_visualization_publisher(
    world,
    ros_node,
)
print(f"RViz fixed frame: {world.root.name}")
print(f"MarkerArray topic: {visualization_publisher.topic_name}")


INFO:viz_marker.py::138 __post_init__ VizMarkerPublisher started. Fixed frame is apartment/apartment_root


RViz fixed frame: apartment/apartment_root
MarkerArray topic: /semworld/viz_marker


In [5]:
assert context.evaluate_conditions
grasp_description = front_grasp(kitchen, Arms.RIGHT)


## Plan one short GCS approach

The route moves the base from its nearby initial position to the same reachable pose used by the compact pouring example. The bowl is already reachable from that pose, so no second route is created.


In [6]:
navigation_graph = create_navigation_graph(
    kitchen,
    bounds=(1.2, 2.0, 1.6, 3.0),
    region_name="kitchen_counter_pouring_navigation_free_space",
)
start_point = navigation_point(kitchen.initial_base_pose, world.root)
pickup_position = Point3(1.7, 2.0, 0.2, reference_frame=world.root)
pickup_base_pose = Pose.from_xyz_rpy(
    x=pickup_position.x,
    y=pickup_position.y,
    yaw=0.0,
    reference_frame=world.root,
)
pickup_route = create_navigation_route(
    navigation_graph,
    start_point,
    pickup_position,
    pickup_base_pose,
    minimum_waypoint_count=2,
)

# The pitcher and bowl sit far enough apart on the counter that no single base
# pose puts both within the right arm's reach (IsObjectReachableBy is false for
# any shared candidate); the robot repositions between picking up and pouring,
# the same way a person would shift their stance to reach a farther spot.
pour_position = Point3(1.9, 2.2, 0.2, reference_frame=world.root)
pour_base_pose = Pose.from_xyz_rpy(
    x=pour_position.x,
    y=pour_position.y,
    yaw=0.0,
    reference_frame=world.root,
)
pour_route = create_navigation_route(
    navigation_graph,
    pickup_position,
    pour_position,
    pour_base_pose,
    minimum_waypoint_count=2,
)

def route_length(route):
    return sum(
        math.hypot(
            float(route_end.x - route_start.x),
            float(route_end.y - route_start.y),
        )
        for route_start, route_end in zip(route.path, route.path[1:])
    )

short_route_length = route_length(pickup_route)
pour_route_length = route_length(pour_route)

{
    "graph_nodes": len(navigation_graph.graph.nodes()),
    "graph_edges": len(navigation_graph.graph.edges()),
    "pickup_route_length_metres": round(short_route_length, 3),
    "pickup_filtered_waypoints": len(pickup_route.path),
    "pour_route_length_metres": round(pour_route_length, 3),
    "pour_filtered_waypoints": len(pour_route.path),
}


INFO:boxes.py::623 free_space_from_semantic_annotation Free space calculated in 18.099364 ms
INFO:boxes.py::642 free_space_from_semantic_annotation Connectivity calculated in 0.309796 ms


{'graph_nodes': 1,
 'graph_edges': 0,
 'pickup_route_length_metres': 0.406,
 'pickup_filtered_waypoints': 2,
 'pour_route_length_metres': 0.283,
 'pour_filtered_waypoints': 2}

In [7]:
plot_navigation_routes(
    navigation_graph,
    [
        ("Pitcher approach", pickup_route, "black"),
        ("Bowl approach", pour_route, "blue"),
    ],
    target=bowl.global_pose.to_position(),
)


In [8]:
pickup_navigation_poses, pickup_navigation_steps = create_navigation_playback(
    pickup_route,
    world.root,
)
pour_navigation_poses, pour_navigation_steps = create_navigation_playback(
    pour_route,
    world.root,
)
len(pickup_navigation_poses), len(pour_navigation_poses)


(6, 4)

## Approach, pick up, and pour

One top-level sequential plan prepares the robot, performs the short approach, picks up the pitcher, and pours into the nearby bowl. No navigation action occurs between pickup and pouring.


In [9]:
pouring_action = PouringAction(
    target_container=bowl,
    source_container=pouring_source,
    arm=Arms.RIGHT,
)
task = sequential(
    [
        ParkArmsAction(Arms.BOTH),
        MoveTorsoAction(TorsoState.HIGH),
        *pickup_navigation_steps,
        PickUpAction(pouring_source, Arms.RIGHT, grasp_description),
        *pour_navigation_steps,
        pouring_action,
    ],
    context=context,
)

with simulated_robot:
    task.perform()


In [10]:
pre_pour_pose, planned_pour_pose = pouring_action._pour_poses()
pre_pour_rotation = Rotation.from_quat(
    [float(value) for value in pre_pour_pose.to_quaternion().to_np()]
)
planned_pour_rotation = Rotation.from_quat(
    [float(value) for value in planned_pour_pose.to_quaternion().to_np()]
)
pouring_tilt_radians = (pre_pour_rotation.inv() * planned_pour_rotation).magnitude()

initial_position = kitchen.initial_base_pose.to_position()
pickup_position_3d = pickup_route.base_pose.to_position()
pour_position_3d = pour_route.base_pose.to_position()
final_position = robot.root.global_pose.to_position()
planned_base_displacement = math.dist(
    [float(initial_position.x), float(initial_position.y)],
    [float(pickup_position_3d.x), float(pickup_position_3d.y)],
)
final_base_error = math.dist(
    [float(final_position.x), float(final_position.y)],
    [float(pour_position_3d.x), float(pour_position_3d.y)],
)

tool_frame = ViewManager.get_end_effector_view(Arms.RIGHT, robot).tool_frame
pitcher_rotation = pitcher.global_pose.to_rotation_matrix().to_np()[:3, :3]
pitcher_up = pitcher_rotation @ np.array([0.0, 0.0, 1.0])
achieved_pitcher_tilt = math.acos(float(np.clip(pitcher_up[2], -1.0, 1.0)))

assert planned_base_displacement < 0.5
assert short_route_length < 0.75
assert pour_route_length < 0.75
assert final_base_error < 1e-6
assert pitcher.parent_connection.parent == tool_frame
assert math.isclose(pouring_tilt_radians, pouring_action.tilt_angle, abs_tol=1e-6)
assert math.isclose(achieved_pitcher_tilt, pouring_tilt_radians, abs_tol=0.02)

{
    "planned_base_displacement_metres": round(planned_base_displacement, 3),
    "gcs_route_length_metres": round(short_route_length, 3),
    "navigation_updates": len(pickup_navigation_poses),
    "navigation_after_pickup": True,
    "pitcher_attached_to_right_gripper": pitcher.parent_connection.parent == tool_frame,
    "pouring_tilt_radians": round(pouring_tilt_radians, 3),
    "achieved_pitcher_tilt_radians": round(achieved_pitcher_tilt, 3),
    "target_container": str(bowl.name),
}


{'planned_base_displacement_metres': 0.406,
 'gcs_route_length_metres': 0.406,
 'navigation_updates': 6,
 'navigation_after_pickup': True,
 'pitcher_attached_to_right_gripper': True,
 'pouring_tilt_radians': np.float64(1.85),
 'achieved_pitcher_tilt_radians': 1.848,
 'target_container': 'bowl.stl'}